In [ ]:
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import LinearProblem
import numpy
import ufl

domain = mesh.create_unit_square(MPI.COMM_WORLD, 8, 8, mesh.CellType.quadrilateral)


V = fem.functionspace(domain, ("Lagrange", 1))


uD = fem.Function(V)
uD.interpolate(lambda x: 1 + x[0] ** 2 + 2 * x[1] ** 2)

tdim = domain.topology.dim
fdim = tdim - 1
domain.topology.create_connectivity(fdim, tdim)
boundary_facets = mesh.exterior_facet_indices(domain.topology)

boundary_dofs = fem.locate_dofs_topological(V, fdim, boundary_facets)
bc = fem.dirichletbc(uD, boundary_dofs)


u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

f = fem.Constant(domain, default_scalar_type(-6))

a = ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = f * v * ufl.dx


problem = LinearProblem(
    a,
    L,
    bcs=[bc],
    petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
    petsc_options_prefix="Poisson",
)
uh = problem.solve()

# import pyvista
# from dolfinx import plot

# domain.topology.create_connectivity(tdim, tdim)
# topology, cell_types, geometry = plot.vtk_mesh(domain, tdim)
# grid = pyvista.UnstructuredGrid(topology, cell_types, geometry)

# plotter = pyvista.Plotter()
# plotter.add_mesh(grid, show_edges=True)
# plotter.view_xy()
# plotter.show()

In [3]:
uD.x.array[:]

array([1.      , 1.03125 , 1.015625, 1.046875, 1.125   , 1.140625,
       1.0625  , 1.09375 , 1.28125 , 1.296875, 1.1875  , 1.140625,
       1.171875, 1.5     , 1.515625, 1.34375 , 1.265625, 1.25    ,
       1.28125 , 1.78125 , 1.796875, 1.5625  , 1.421875, 1.375   ,
       1.390625, 1.421875, 2.125   , 2.140625, 1.84375 , 1.640625,
       1.53125 , 1.515625, 1.5625  , 1.59375 , 2.53125 , 2.546875,
       2.1875  , 1.921875, 1.75    , 1.671875, 1.6875  , 1.765625,
       1.796875, 3.      , 3.015625, 2.59375 , 2.265625, 2.03125 ,
       1.890625, 1.84375 , 1.890625, 2.      , 2.03125 , 3.0625  ,
       2.671875, 2.375   , 2.171875, 2.0625  , 2.046875, 2.125   ,
       3.140625, 2.78125 , 2.515625, 2.34375 , 2.265625, 2.28125 ,
       3.25    , 2.921875, 2.6875  , 2.546875, 2.5     , 3.390625,
       3.09375 , 2.890625, 2.78125 , 3.5625  , 3.296875, 3.125   ,
       3.765625, 3.53125 , 4.      ])

In [4]:
u_topology, u_cell_types, u_geometry = plot.vtk_mesh(V)
u_grid = pyvista.UnstructuredGrid(u_topology, u_cell_types, u_geometry)
u_grid.point_data["u"] = uh.x.array.real
u_grid.set_active_scalars("u")
u_plotter = pyvista.Plotter()
u_plotter.add_mesh(u_grid, show_edges=True)
u_plotter.view_xy()
u_plotter.show()

NameError: name 'plot' is not defined

In [ ]:
import numpy as np
from mpi4py import MPI
from dolfinx import mesh
import basix.ufl

points = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [1.0, 1.0],
    [0.0, 1.0],
], dtype=np.float64)

cells = np.array([
    [0, 1, 2],
    [0, 2, 3],
], dtype=np.int64)

element = basix.ufl.element(
    basix.ElementFamily.P,
    "triangle",
    1,
    shape=(2,),
)

domain = mesh.create_mesh(
    MPI.COMM_WORLD,
    cells,
    element,
    points,
)

In [ ]:
import qewton

X = qewton.Variable("x", dim=2)
rectanlge = qewton.geometries.Rectangle(X, [0.0, 0.0], 1.0, 1.0, backend=qewton.backends.NumPyBackend)
mesh_domain = rectanlge.create_mesh(0.1)

In [ ]:
import numpy as np
from mpi4py import MPI
from dolfinx import mesh
import basix.ufl
from mpi4py import MPI
from dolfinx import mesh, fem, default_scalar_type
from dolfinx.fem.petsc import LinearProblem
import ufl

element = basix.ufl.element(
    basix.ElementFamily.P,
    "triangle",
    1,
    shape=(2,),
)

domain = mesh.create_mesh(
    MPI.COMM_WORLD,
    mesh_domain.mesh.cells,
    element,
    mesh_domain.mesh.vertices,
)
V = fem.functionspace(domain, ("Lagrange", 1))


uD = fem.Function(V)
uD.interpolate(lambda x: 1 + x[0] ** 2 + 2 * x[1] ** 2)

tdim = domain.topology.dim
fdim = tdim - 1
domain.topology.create_connectivity(fdim, tdim)
boundary_facets = mesh.exterior_facet_indices(domain.topology)

boundary_dofs = fem.locate_dofs_topological(V, fdim, boundary_facets)
bc = fem.dirichletbc(uD, boundary_dofs)


u = ufl.TrialFunction(V)
v = ufl.TestFunction(V)

f = fem.Constant(domain, default_scalar_type(-6))

a = ufl.dot(ufl.grad(u), ufl.grad(v)) * ufl.dx
L = f * v * ufl.dx


problem = LinearProblem(
    a,
    L,
    bcs=[bc],
    petsc_options={"ksp_type": "preonly", "pc_type": "lu"},
    petsc_options_prefix="Poisson",
)
uh = problem.solve()